# Tokenization — Split + Tokenize + Save

**Input:** `data/model_ready/` encoded CSVs + `reports/recommended_tokenization.json`
**Output:** `data/tokenized/` HuggingFace Arrow datasets (train/validation/test per task)

In [1]:
import json
import os
import shutil
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [2]:
# ── Configuration ──
MODEL_NAME = "answerdotai/ModernBERT-base"
DATASET_FOLDER = "data/model_ready"
OUTPUT_FOLDER = "data/tokenized"
JSON_PATH = "reports/recommended_tokenization.json"

TEST_SIZE = 0.15
VALID_SIZE = 0.15
RANDOM_STATE = 42

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Configuration set")
print(f"  Model        : {MODEL_NAME}")
print(f"  Input folder : {DATASET_FOLDER}")
print(f"  Output folder: {OUTPUT_FOLDER}")
print(f"  Split        : {1-TEST_SIZE-VALID_SIZE:.0%} train / {VALID_SIZE:.0%} val / {TEST_SIZE:.0%} test")

Configuration set
  Model        : answerdotai/ModernBERT-base
  Input folder : data/model_ready
  Output folder: data/tokenized
  Split        : 70% train / 15% val / 15% test


In [3]:
# ── Load recommended max_length from analysis notebook ──
with open(JSON_PATH, "r") as f:
    _config = json.load(f)

recommended_lengths = {task: cfg["max_length"] for task, cfg in _config.items()}

print("Loaded recommended_tokenization.json")
for task, cfg in _config.items():
    print(f"  {task:<15s}  max_length={cfg['max_length']}  truncate={cfg['truncate_pct']:.2f}%")

Loaded recommended_tokenization.json
  intent           max_length=128  truncate=1.15%
  root_cause       max_length=256  truncate=1.16%
  sentiment        max_length=128  truncate=1.41%
  risk             max_length=256  truncate=1.16%


In [ ]:
# ── Load encoded datasets ──
intent_df = pd.read_csv(os.path.join(DATASET_FOLDER, "intent_encoded.csv"))
sentiment_df = pd.read_csv(os.path.join(DATASET_FOLDER, "sentiment_encoded.csv"))
root_df = pd.read_csv(os.path.join(DATASET_FOLDER, "root_cause_encoded.csv"))
risk_df = pd.read_csv(os.path.join(DATASET_FOLDER, "risk_dataset.csv"))

# Remove obvious noise / unclear labels before splitting
intent_df = intent_df[intent_df["intent"] != "noise"].copy()
root_df = root_df[root_df["root_cause"] != "other"].copy()

# Remove classes with fewer than 10 samples before stratified split
# This prevents sklearn ValueError when a class has only 1 row

def drop_small_classes(df, col_name, min_count=10):
    counts = df[col_name].value_counts()
    rare_labels = counts[counts < min_count].index.tolist()

    if len(rare_labels) > 0:
        print(f"Dropping rare labels in {col_name}: {rare_labels}")
        df = df[~df[col_name].isin(rare_labels)].copy()

    return df

# Apply to the dataset columns that are sensitive to stratified splits
intent_df = drop_small_classes(intent_df, "intent", min_count=10)
root_df = drop_small_classes(root_df, "root_cause", min_count=10)

print("Datasets loaded")
print(f"  intent     : {len(intent_df):>7,} rows")
print(f"  sentiment  : {len(sentiment_df):>7,} rows")
print(f"  root_cause : {len(root_df):>7,} rows")
print(f"  risk       : {len(risk_df):>7,} rows")

Datasets loaded
  intent     :  20,043 rows
  sentiment  :  16,432 rows
  root_cause :  11,594 rows
  risk       :  11,594 rows


In [6]:
# ── Verify required columns exist ──
REQUIRED = {
    "intent":     (intent_df,    ["text", "intent",     "intent_label"]),
    "sentiment":  (sentiment_df, ["text", "sentiment",  "sentiment_label"]),
    "root_cause": (root_df,      ["text", "root_cause", "root_cause_label"]),
    "risk":       (risk_df,      ["text", "risk_band",  "risk_label"]),
}

all_ok = True
for task, (df, cols) in REQUIRED.items():
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"FAIL {task}: missing columns {missing}")
        print(f"     Found: {sorted(df.columns.tolist())}")
        all_ok = False
    else:
        print(f"OK   {task}: all columns present")

if not all_ok:
    raise ValueError("Column check failed. Fix the CSV files first.")


OK   intent: all columns present
OK   sentiment: all columns present
OK   root_cause: all columns present
OK   risk: all columns present


In [10]:
# ── Split all datasets: 70% train / 15% val / 15% test ──
# Pass 1: full -> train_val (85%) + test (15%)
# Pass 2: train_val -> train (70%) + val (15%)

TASKS = {
    "intent":     {"df": intent_df,    "label_col": "intent_label"},
    "sentiment":  {"df": sentiment_df, "label_col": "sentiment_label"},
    "root_cause": {"df": root_df,      "label_col": "root_cause_label"},
    "risk":       {"df": risk_df,      "label_col": "risk_label"},
}

splits = {}  # splits[task] = {"train": df, "val": df, "test": df}

print(f"{'Task':<12s}  {'Train':>7s}  {'Val':>7s}  {'Test':>7s}  {'Total':>7s}")
print(f"{'-'*12}  {'-'*7}  {'-'*7}  {'-'*7}  {'-'*7}")

for task_name, task_info in TASKS.items():
    df = task_info["df"].copy()
    label_col = task_info["label_col"]

    # Safety check: some label classes may still be too rare even after cleanup
    counts = df[label_col].value_counts()
    rare_labels = counts[counts < 2].index.tolist()
    if rare_labels:
        print(f"Warning: {task_name} still has classes with < 2 rows after cleanup: {rare_labels}")
        print("Dropping them before stratified split.")
        df = df[~df[label_col].isin(rare_labels)].copy()

    # Pass 1: split off test set
    train_val, test = train_test_split(
        df, test_size=TEST_SIZE, stratify=df[label_col], random_state=RANDOM_STATE
    )

    # Pass 2: split train_val into train + val
    adjusted_valid = VALID_SIZE / (1.0 - TEST_SIZE)
    train, val = train_test_split(
        train_val, test_size=adjusted_valid, stratify=train_val[label_col], random_state=RANDOM_STATE
    )

    splits[task_name] = {"train": train, "val": val, "test": test}
    total = len(train) + len(val) + len(test)
    print(f"{task_name:<12s}  {len(train):>7,}  {len(val):>7,}  {len(test):>7,}  {total:>7,}")

Task            Train      Val     Test    Total
------------  -------  -------  -------  -------
intent         14,029    3,007    3,007   20,043
sentiment      11,502    2,465    2,465   16,432
Dropping them before stratified split.
root_cause      8,115    1,739    1,739   11,593
risk            8,115    1,739    1,740   11,594


In [11]:
# ── Load tokenizer once ──
print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer ready (vocab size: {tokenizer.vocab_size:,}, model max: {tokenizer.model_max_length})")

# Verify recommended lengths don't exceed model limit
for task, length in recommended_lengths.items():
    if length > tokenizer.model_max_length:
        raise ValueError(f"{task}: recommended max_length ({length}) exceeds model limit ({tokenizer.model_max_length})")
    print(f"  OK  {task:<12s}  max_length={length}")

Loading tokenizer: answerdotai/ModernBERT-base ...
Tokenizer ready (vocab size: 50,280, model max: 8192)
  OK  intent        max_length=128
  OK  root_cause    max_length=256
  OK  sentiment     max_length=128
  OK  risk          max_length=256


In [12]:
# ── Tokenize all tasks and save to disk ──

TASK_CONFIGS = {
    "intent":     {"text_col": "text", "label_col": "intent_label"},
    "sentiment":  {"text_col": "text", "label_col": "sentiment_label"},
    "root_cause": {"text_col": "text", "label_col": "root_cause_label"},
    "risk":       {"text_col": "text", "label_col": "risk_label"},
}

for task_name, config in TASK_CONFIGS.items():
    max_len = recommended_lengths[task_name]
    text_col = config["text_col"]
    label_col = config["label_col"]
    
    print(f"\nTokenizing {task_name.upper()} (max_length={max_len}) ...")
    
    split_datasets = {}
    
    for split_name in ["train", "val", "test"]:
        split_df = splits[task_name][split_name].reset_index(drop=True)
        texts = split_df[text_col].astype(str).tolist()
        labels = split_df[label_col].tolist()
        
        # Tokenize in batches of 512
        all_input_ids = []
        all_attention_mask = []
        
        for start in range(0, len(texts), 512):
            batch = texts[start : start + 512]
            enc = tokenizer(
                batch,
                max_length=max_len,
                truncation=True,
                padding=False,        # DataCollatorWithPadding handles this during training
                return_tensors=None,
            )
            all_input_ids.extend(enc["input_ids"])
            all_attention_mask.extend(enc["attention_mask"])
        
        ds = Dataset.from_dict({
            "input_ids": all_input_ids,
            "attention_mask": all_attention_mask,
            "labels": labels,  # must be 'labels' (with s) for HuggingFace Trainer
        })
        
        hf_split = "validation" if split_name == "val" else split_name
        split_datasets[hf_split] = ds
    
    dd = DatasetDict(split_datasets)
    
    # Save to disk (clear old data first)
    save_path = os.path.join(OUTPUT_FOLDER, task_name)
    if os.path.exists(save_path):
        shutil.rmtree(save_path)
    dd.save_to_disk(save_path)
    
    print(f"  Saved {task_name} -> {save_path}")
    print(f"    train={len(dd['train']):,}  val={len(dd['validation']):,}  test={len(dd['test']):,}")

print("\nAll four datasets tokenized and saved.")
print("\nTo load in training notebook:")
print("  from datasets import DatasetDict")
print(f"  dd = DatasetDict.load_from_disk('{OUTPUT_FOLDER}/intent')")
print("  # dd['train'], dd['validation'], dd['test']")


Tokenizing INTENT (max_length=128) ...


Saving the dataset (0/1 shards):   0%|          | 0/14029 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3007 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3007 [00:00<?, ? examples/s]

  Saved intent -> data/tokenized\intent
    train=14,029  val=3,007  test=3,007

Tokenizing SENTIMENT (max_length=128) ...


...

  Saved root_cause -> data/tokenized\root_cause
    train=8,115  val=1,739  test=1,739

Tokenizing RISK (max_length=256) ...


Saving the dataset (0/1 shards):   0%|          | 0/8115 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1739 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1740 [00:00<?, ? examples/s]

  Saved risk -> data/tokenized\risk
    train=8,115  val=1,739  test=1,740

All four datasets tokenized and saved.

To load in training notebook:
  from datasets import DatasetDict
  dd = DatasetDict.load_from_disk('data/tokenized/intent')
  # dd['train'], dd['validation'], dd['test']
